## import libraries

In [1]:
import ee
from dotenv import load_dotenv
import os
import ee
import pandas as pd
from pathlib import Path

## initialize google earth engine

In [2]:
ee.Authenticate()

True

In [3]:
load_dotenv()  
project = os.getenv("PROJECT")
ee.Initialize(project=project)

## read datasets

In [4]:
root_embeddings = Path("_data/embeddings_exports")
root_dataset = Path("_data/crop_country_exports")

In [12]:
def build_files_dataframe(embeddings, dataset):
    rows = []

    folders = [dataset, embeddings]

    for folder in folders:
        for file in folder.glob("*"):
    
            if file.suffix not in [".csv", ".parquet"]:
                continue
            name = file.stem  
            parts = name.split("_")

            if "embedding" in parts:
                year = parts[-2]
                country = parts[-3]
                crop = "_".join(parts[:-3])
                file_type = "embedding"
            else:
                year = parts[-1]
                country = parts[-2]
                crop = "_".join(parts[:-2])
                file_type = "dataset"
            
            rows.append({
                "path": str(file),
                "crop": crop,
                "country": country,
                "year": int(year),
                "type": file_type
            })

    return pd.DataFrame(rows)

In [13]:
files = build_files_dataframe(root_embeddings, root_dataset)
files.head()

,path,crop,country,year,type
0,_data/crop_country_exports/maize_corn_popcorn_...,maize_corn_popcorn,pt,2018,dataset
1,_data/crop_country_exports/maize_corn_popcorn_...,maize_corn_popcorn,pt,2019,dataset
2,_data/crop_country_exports/potatoes_bg_2018.csv,potatoes,bg,2018,dataset
3,_data/crop_country_exports/winter_barley_dk_20...,winter_barley,dk,2018,dataset
4,_data/crop_country_exports/winter_barley_be_20...,winter_barley,be,2016,dataset


In [14]:
df_datasets = files[files['type'] == 'dataset'].drop(columns=['type'])
df_embeddings = files[files['type'] == 'embedding'].drop(columns=['type'])

pairs = pd.merge(df_datasets, 
                 df_embeddings, 
                 on=['crop', 'country', 'year'], 
                 suffixes=('_dataset', '_embedding'))

pairs.head()

,path_dataset,crop,country,year,path_embedding
0,_data/crop_country_exports/maize_corn_popcorn_...,maize_corn_popcorn,be,2019,_data/embeddings_exports/maize_corn_popcorn_be...
1,_data/crop_country_exports/maize_corn_popcorn_...,maize_corn_popcorn,at,2019,_data/embeddings_exports/maize_corn_popcorn_at...


In [15]:
datasets_dict = {}
embeddings_dict = {}

for _, row in pairs.iterrows():
    key = f"{row['crop']}_{row['country']}_{row['year']}"
    
    datasets_dict[key] = pd.read_csv(row['path_dataset'])
    embeddings_dict[key] = pd.read_parquet(row['path_embedding'])

print(f"Successfully loaded {len(datasets_dict)} pairs.")
print(f"Dictionaries keys are  {datasets_dict.keys()} pairs.")

Successfully loaded 2 pairs.
Dictionaries keys are  dict_keys(['maize_corn_popcorn_be_2019', 'maize_corn_popcorn_at_2019']) pairs.


In [16]:
datasets_dict['maize_corn_popcorn_be_2019'].head(2)

,country_id,year,hcat4_code,hcat4_name,long,lat
0,be,2019,3310106000,maize_corn_popcorn,5.140735,50.457525
1,be,2019,3310106000,maize_corn_popcorn,4.351956,50.257848


In [17]:
embeddings_dict['maize_corn_popcorn_be_2019'].head(2)

,tile_lon,tile_lat,pixel_row,pixel_col,crs,embedding,long_lat,crop,country_id,year
0,5.15,50.45,484,302,EPSG:32631,"[5.015644, -0.31347775, -0.9404332, 4.3259926,...","[5.140735275624097, 50.45752549909366]",maize_corn_popcorn,be,2019
1,4.35,50.25,474,378,EPSG:32631,"[5.2835274, 1.1251957, 1.418725, 3.4734302, 1....","[4.3519556147586504, 50.25784835964254]",maize_corn_popcorn,be,2019


In [ ]:
# maize_keys = [k for k in datasets_dict.keys() if "be" in k]
# maize_keys

## get point images

| Data Type   | Access Method                     | Description                                                |
|--------------|----------------------------------|------------------------------------------------------------|
| Dataset     | `datasets_dict[key].iloc[i]`     | The i-th row of tabular data (labels, etc.).              |
| Embeddings   | `embeddings_dict[key].iloc[i]`   | The i-th row of the embedding vector.                     |
| Satellite    | `gee_dict[key][i]['s2']`         | The full Sentinel-2 time series for that exact point.     |

In [18]:
def to_df(raw_data):
    df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    df['date'] = pd.to_datetime(df['time'], unit='ms')
    return df.drop(columns=['id', 'time'])

In [19]:
def get_sits_data(lat, lon, year):
    poi = ee.Geometry.Point([lon, lat])
    start = f'{year}-01-01'
    end = f'{year}-12-31'

    # 1. Sentinel-2 (All 10 requested bands)
    s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
    s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterBounds(poi)
              .filterDate(start, end)
              #.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))
              .select(s2_bands))

    # 2. Sentinel-1 (VV and VH)
    s1_col = (ee.ImageCollection("COPERNICUS/S1_GRD")
              .filterBounds(poi)
              .filterDate(start, end)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .select(['VV', 'VH']))

    # 3. Extracting Point Data (The fast part)
    # getRegion returns a list of lists: [id, lon, lat, time, band1, band2...]
    s2_raw = s2_col.getRegion(poi, 10).getInfo()
    s1_raw = s1_col.getRegion(poi, 10).getInfo()

    return to_df(s2_raw), to_df(s1_raw)

# Example call
# df_s2, df_s1 = get_sits_data(47.658, -2.759, 2023)

In [20]:
gee_dict = {}

for key in datasets_dict.keys():
    df_main = datasets_dict[key]
    year = int(key.split('_')[-1])
    
    # Initialize a list to hold results for every point in this specific file
    file_results = []
    
    print(f"Starting extraction for {key} ({len(df_main)} points)...")
    
    for idx, row in df_main.iterrows():
        lat, lon = row['lat'], row['long']
        
        try:
            # Fetch S1 and S2 time series for this specific point
            df_s2, df_s1 = get_sits_data(lat, lon, year)
            
            # Store them as a pair
            file_results.append({
                's2': df_s2,
                's1': df_s1,
                'point_coord': (lat, lon) # Helpful for double-checking later
            })
        except Exception as e:
            print(f"Error at row {idx} in {key}: {e}")
            file_results.append(None) # Keep the index aligned even on failure
            
    gee_dict[key] = file_results

Starting extraction for maize_corn_popcorn_be_2019 (500 points)...
Starting extraction for maize_corn_popcorn_at_2019 (500 points)...


In [21]:
gee_dict.keys()

dict_keys(['maize_corn_popcorn_be_2019', 'maize_corn_popcorn_at_2019'])

In [34]:
gee_dict['maize_corn_popcorn_at_2019'][7]['s1']

,longitude,latitude,VV,VH,date
0,16.193795,48.473407,-4.938481,-16.804854,2019-01-02 05:09:54
1,16.193795,48.473407,-4.633760,-16.341506,2019-01-02 05:09:56
2,16.193795,48.473407,-14.431751,-29.831453,2019-01-05 16:42:49
3,16.193795,48.473407,-9.663594,-19.940700,2019-01-09 05:01:53
4,16.193795,48.473407,-15.544306,-22.543593,2019-01-10 16:50:52
...,...,...,...,...,...
236,16.193795,48.473407,-17.760286,-23.734001,2019-12-18 16:50:19
237,16.193795,48.473407,-7.747447,-20.101495,2019-12-22 05:09:11
238,16.193795,48.473407,-9.672873,-16.513034,2019-12-25 16:42:14
239,16.193795,48.473407,-13.383955,-25.113216,2019-12-29 05:01:12


In [36]:
import pandas as pd
import pickle
from pathlib import Path

# 1. Create the directory if it doesn't exist
gee_output_dir = Path("_data/gee_exports")
gee_output_dir.mkdir(parents=True, exist_ok=True)

# 2. Saving the dictionary
def save_gee_data(data_dict, folder):
    for key, content in data_dict.items():
        file_path = folder / f"{key}_gee.pkl"
        with open(file_path, 'wb') as f:
            pickle.dump(content, f)
    print(f"Successfully saved {len(data_dict)} files to {folder}")

# Execute the save
save_gee_data(gee_dict, gee_output_dir)

Successfully saved 2 files to _data/gee_exports


In [37]:
# To Load (next time you open your script)
with open('_data/gee_exports/maize_corn_popcorn_at_2019_gee.pkl', 'rb') as f:
    ooo = pickle.load(f)

In [42]:
ooo[0].keys()

dict_keys(['s2', 's1', 'point_coord'])